In [75]:
import pandas as pd
import numpy as np
# Load dữ liệu Excel
df = pd.read_excel("cafe_data.xlsx")

df.head()

,Dấu thời gian,ten_quan,doi_tuong,muc_dich,tim_kiem,wifi,khong_gian,o_cam,gia_ca,ngon,view
0,2026-05-22 23:40:59.507,"Sora Coffee -Ngã Ba, Nam Kỳ Khởi Nghĩa Huỳnh L...",Freelancer,Hẹn hò,Tiktok,4,3,2,4,4,5
1,2026-05-22 23:49:58.369,"Sora Coffee -Ngã Ba, Nam Kỳ Khởi Nghĩa Huỳnh L...",Freelancer,Làm việc,Google Maps,5,3,3,4,5,4
2,2026-05-22 23:52:09.179,"SH Flow -Trịnh Hoài Đức, Điện Bàn Đông",Freelancer,Làm việc,Google Maps,4,5,4,4,3,2
3,2026-05-22 23:54:54.047,"Sora Coffee -Ngã Ba, Nam Kỳ Khởi Nghĩa Huỳnh L...",Freelancer,Làm việc,Tiktok,4,3,3,4,5,4
4,2026-05-23 16:02:23.634,"Túc Tắc Tea -76 Huỳnh Văn Nghệ, Ngũ Hành Sơn",Freelancer,Làm việc,Bạn bè giới thiệu,3,5,5,1,2,1


In [76]:
features = ["wifi", "khong_gian", "o_cam", "gia_ca", "ngon", "view"]

coffee_features = df.groupby("ten_quan")[features].mean().reset_index()

coffee_features.head()

,ten_quan,wifi,khong_gian,o_cam,gia_ca,ngon,view
0,"LEO Cà phê -Lô 12 Huỳnh Văn Nghệ, Ngũ Hành Sơn",3.978261,1.934783,2.097826,2.184783,3.847826,4.086957
1,"Milano Coffee- Huỳnh Lắm, Ngũ Hành Sơn",4.562500,2.072917,1.593750,2.187500,3.989583,2.458333
2,"SH Flow -Trịnh Hoài Đức, Điện Bàn Đông",3.953271,4.570093,4.476636,4.065421,2.934579,1.925234
3,"Sora Coffee -Ngã Ba, Nam Kỳ Khởi Nghĩa Huỳnh L...",4.467213,2.500000,1.991803,4.204918,4.172131,4.524590
4,"ToCha- 65 -Lê Thiện Trị, Ngũ Hành Sơn",4.540682,2.036745,1.488189,2.070866,3.981627,2.002625


In [77]:
weights = {
    "hoc tap": {
        "wifi": 0.30,
        "khong_gian": 0.25,
        "o_cam": 0.25,
        "gia_ca": 0.05,
        "ngon": 0.10,
        "view": 0.05
    },
    "lam viec": {
        "wifi": 0.30,
        "o_cam": 0.25,
        "khong_gian": 0.20,
        "gia_ca": 0.10,
        "ngon": 0.10,
        "view": 0.05
    },
    "thu gian": {
        "view": 0.30,
        "khong_gian": 0.25,
        "ngon": 0.20,
        "wifi": 0.10,
        "gia_ca": 0.10,
        "o_cam": 0.05
    },
    "gap ban be": {
        "khong_gian": 0.25,
        "ngon": 0.25,
        "view": 0.20,
        "wifi": 0.10,
        "gia_ca": 0.10,
        "o_cam": 0.10
    },
    "hen ho": {
        "view": 0.35,
        "khong_gian": 0.30,
        "ngon": 0.15,
        "gia_ca": 0.10,
        "wifi": 0.05,
        "o_cam": 0.05
    },
    "khac": {
        "wifi": 0.20,
        "khong_gian": 0.20,
        "o_cam": 0.15,
        "gia_ca": 0.15,
        "ngon": 0.15,
        "view": 0.15
    }
}

In [78]:
def compute_score(user, coffee, w):
    max_score = 0
    score = 0

    for i, f in enumerate(features):
        diff = abs(user[i] - coffee[i])
        score += w[f] * diff
        max_score += w[f] * 4   # max diff = 4

    score = 1 - (score / max_score)

    return score

In [79]:
def recommend_top3(user_input, purpose):

    w = weights.get(purpose, weights["khac"])

    user_vector = [
        user_input["wifi"],
        user_input["khong_gian"],
        user_input["o_cam"],
        user_input["gia_ca"],
        user_input["ngon"],
        user_input["view"]
    ]

    results = []

    for _, row in coffee_features.iterrows():

        coffee_vector = [
            row["wifi"],
            row["khong_gian"],
            row["o_cam"],
            row["gia_ca"],
            row["ngon"],
            row["view"]
        ]

        score = compute_score(user_vector, coffee_vector, w)

        results.append([row["ten_quan"], score])

    result_df = pd.DataFrame(results, columns=["ten_quan", "score"])

    return result_df.sort_values("score", ascending=False).head(3)

In [ ]:
user_input = {
    "wifi": 4,
    "khong_gian": 2,
    "o_cam": 2,
    "gia_ca": 2,
    "ngon": 4,
    "view": 4   
}

result = recommend_top3(user_input, "thu gian")


for _, row in result.iterrows():
    print(f"{row['ten_quan']} | Score: {row['score']:.4f}")

LEO Cà phê -Lô 12 Huỳnh Văn Nghệ, Ngũ Hành Sơn | Score: 0.9754
Xeko -21 Huỳnh Văn Nghệ, Ngũ Hành Sơn | Score: 0.8976
Túc Tắc Tea -76 Huỳnh Văn Nghệ, Ngũ Hành Sơn | Score: 0.8857
